Installations

In [1]:
import pandas as pd
import pytz
import re
import unicodedata
import nltk
import json
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
from nltk.corpus import stopwords
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.sparse import csr_matrix
import os
import random
import sqlite3
import time
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.chains.summarize import load_summarize_chain
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import adjusted_rand_score
import ast

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\fatim\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\fatim\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\fatim\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
from data.init_db import (
    upsert_clustering_run, insert_cluster, 
    map_comments_to_cluster, insert_valley_data, get_connection
)

def clear_old_run_data(run_id):
    conn = sqlite3.connect('data/pdam.db')
    cur = conn.cursor()
    cur.execute("PRAGMA foreign_keys = ON")
    cur.execute("DELETE FROM comment_cluster_map WHERE run_id = ?", (run_id,))
    cur.execute("DELETE FROM clusters WHERE run_id = ?", (run_id,))
    cur.execute("DELETE FROM valley_tracing_data WHERE run_id = ?", (run_id,))
    conn.commit()
    conn.close()

load_dotenv()

True

In [3]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

In [ ]:
MIN_CLUSTER_SIZE = 3
MAX_COMMENTS_PER_CLUSTER = 15

In [5]:
DOMAIN_CONTEXT = "Konteks: Komentar Instagram PDAM Surya Sembada, perusahaan penyedia air bersih Surabaya."

map_prompt = PromptTemplate(
    template="""Kamu adalah analis teks. Berikut adalah komentar pelanggan PDAM Surya Sembada Surabaya:
\"{text}\"
Kata kunci kelompok ini: {keywords}
Tulis ringkasan singkat (1-2 kalimat) dalam bahasa Indonesia yang menangkap inti komentar-komentar ini. Gunakan kata kunci sebagai panduan topik.
RINGKASAN:""",
    input_variables=["text", "keywords"]
)

combine_prompt = PromptTemplate(
    template="""Kamu adalah analis teks. Berikut adalah ringkasan dari beberapa komentar pelanggan PDAM Surya Sembada Surabaya:
\"{text}\"
Gabungkan menjadi satu ringkasan akhir (1-2 kalimat) dalam bahasa Indonesia.
RINGKASAN AKHIR:""",
    input_variables=["text"]
)

reduce_prompt = PromptTemplate(
    template="""Kamu adalah analis teks. {domain_context}
Berikut adalah ringkasan dari setiap kelompok komentar pelanggan PDAM:
\"{text}\"
Tulis 3-5 kalimat dalam satu paragraf yang menggambarkan topik-topik utama dan opini umum masyarakat.
Aturan: mulai kalimat pertama dengan "Warga..." atau topik paling dominan; variasikan awal tiap kalimat; tanpa kalimat pembuka atau penutup; hanya fakta dari ringkasan di atas; jangan mengarang.
RINGKASAN GLOBAL:""",
    input_variables=["text", "domain_context"]
)

theme_prompt = PromptTemplate(
    template="""Kamu adalah analis teks. {domain_context}
Komentar dari satu kelompok pelanggan PDAM:
{responses}
Kata kunci kelompok: {keywords}
Tentukan tema utama dalam 2-5 kata bahasa Indonesia berdasarkan komentar DAN kata kunci di atas.
Gunakan nama wilayah jika kata kunci mengandung nama tempat spesifik (contoh: "Gangguan Air Sememi", "Mati Air Karah").
Jika tidak ada tema jelas, tulis "Misc / Minoritas".
Tulis HANYA tema tanpa tanda kutip dan tanpa penjelasan.
TEMA UTAMA:""",
    input_variables=["responses", "keywords", "domain_context"]
)

commnets

In [2]:
comments = pd.read_csv('instagram_comments.csv')
data = comments.loc[:,['post_id','comment_id','created_at','username','text','parent_comment_id']]
data['comments'] = data['text']
data['timestamp'] = data['created_at']
data

,post_id,comment_id,created_at,username,text,parent_comment_id,comments,timestamp
0,DQ51a5oEw2y,18075080645463218,1762841875,heframid,Selamat ulang tahun ke-36 Harian Surya! 🎉 Semo...,NaN,Selamat ulang tahun ke-36 Harian Surya! 🎉 Semo...,1762841875
1,DQ51a5oEw2y,18540359950061107,1762844716,pdamsuryasembada,@heframid Terima kasih Heframid. Semoga sukses...,NaN,@heframid Terima kasih Heframid. Semoga sukses...,1762844716
2,DQ51a5oEw2y,18542961421052542,1762868957,kharistiano28,👏👏👏 selalu menjalin kerjasama yg baik dgn medi...,NaN,👏👏👏 selalu menjalin kerjasama yg baik dgn medi...,1762868957
3,DQ51a5oEw2y,18097328239697834,1762904139,pdamsuryasembada,@kharistiano28 terima kasih Bpk Kharistiano at...,NaN,@kharistiano28 terima kasih Bpk Kharistiano at...,1762904139
4,DQ5v5UsE4As,18005382170826298,1762834417,mhmdrizkyy12._,Selamatt dan semangattt indonesiaku😍🔥,NaN,Selamatt dan semangattt indonesiaku😍🔥,1762834417
...,...,...,...,...,...,...,...,...
27651,Czvi08FPNgg,18064810039417636,1700223629,pdamsuryasembada,@pratyaharasamadhi selamat malam pak mhn infor...,NaN,@pratyaharasamadhi selamat malam pak mhn infor...,1700223629
27652,Czvi08FPNgg,18395681002021119,1700223639,pdamsuryasembada,"@hellow_vee92 Selamat malam pak/bu, mohon maaf...",NaN,"@hellow_vee92 Selamat malam pak/bu, mohon maaf...",1700223639
27653,Czvi08FPNgg,17969390084643039,1700223644,pdamsuryasembada,"@hellow_vee92 Selamat malam pak/bu, mohon maaf...",NaN,"@hellow_vee92 Selamat malam pak/bu, mohon maaf...",1700223644
27654,Czvi08FPNgg,18222529924301128,1700223839,pdamsuryasembada,"@vii.4dity Selamat malam pak/bu, mohon maaf at...",NaN,"@vii.4dity Selamat malam pak/bu, mohon maaf at...",1700223839


Drop empty comments
Remove comments from pdamsuryasembada
Change date from epoch miliseconds
Remove taggings from comments

In [3]:
data = data[data['username'].str.contains('pdamsuryasembada') == False]

data = data.drop_duplicates(subset=['text'])

local_timezone = pytz.timezone('Asia/Jakarta')
data['created_at'] = pd.to_datetime(data['created_at'], unit='s')
data['created_at'] = data['created_at'].dt.tz_localize('UTC').dt.tz_convert(local_timezone)
data['created_at'] = data['created_at'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

def remove_tags(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r'@\w[\w\.]*', '', text).strip()

data['text'] = data['text'].apply(remove_tags)

data = data.dropna(subset=["text"])
data = data[data["text"].astype(str).str.strip() != ""]

In [5]:
data[['timestamp', 'created_at']].rename(columns={'timestamp': 'Sebelum', 'created_at': 'Sesudah'}).head(20)

,Sebelum,Sesudah
0,1762841875,2025-11-11 13:17:55 WIB
2,1762868957,2025-11-11 20:49:17 WIB
4,1762834417,2025-11-11 11:13:37 WIB
6,1762829337,2025-11-11 09:48:57 WIB
9,1762851858,2025-11-11 16:04:18 WIB
10,1762859854,2025-11-11 18:17:34 WIB
11,1762867023,2025-11-11 20:17:03 WIB
15,1762904844,2025-11-12 06:47:24 WIB
17,1762953584,2025-11-12 20:19:44 WIB
19,1762829373,2025-11-11 09:49:33 WIB


cleaning

In [6]:
PUNCT_TO_REMOVE = "!#$%&()*+,./:;<=>@[\\]^_{|}~`"
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', PUNCT_TO_REMOVE))

data["text"] = data["text"].apply(lambda text: remove_punctuation(text))

def remove_emojis(text):
    if not isinstance(text, str):
        return text

    emoji_pattern = re.compile(
        "["                     
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        u"\U0001F700-\U0001F77F"
        u"\U0001F780-\U0001F7FF"
        u"\U0001F800-\U0001F8FF"
        u"\U0001F900-\U0001F9FF"
        u"\U0001FA70-\U0001FAFF"
        u"\u2600-\u26FF"
        u"\u2700-\u27BF"
        u"\ufe0f"
        "]+",
        flags=re.UNICODE,
    )

    return emoji_pattern.sub(r'', text)

data['text'] = data['text'].apply(remove_emojis)
data['text'] = data['text'].str.strip()

def normalize_fancy_unicode(text):
    if not isinstance(text, str):
        return text

    normalized = unicodedata.normalize("NFKD", text)

    cleaned = "".join(
        ch for ch in normalized
        if not unicodedata.combining(ch)
    )
    return cleaned

data["text"] = data["text"].apply(normalize_fancy_unicode)

data = data.dropna(subset=["text"])
data = data[data["text"].astype(str).str.strip() != ""]


case folding

In [9]:
data["text"] = data["text"].str.lower()
data_comments = data["text"]
with pd.option_context('display.max_colwidth', None):
    display(data[['comments', 'text']].rename(columns={'comments': 'Sebelum', 'text': 'Sesudah'}).head(10))

,Sebelum,Sesudah
0,Selamat ulang tahun ke-36 Harian Surya! 🎉 Semoga terus menjadi sumber informasi terpercaya dan inspiratif bagi masyarakat. Terima kasih juga atas kolaborasi positif bersama Perumda Air Minum Surya Sembada!,selamat ulang tahun ke-36 harian surya semoga terus menjadi sumber informasi terpercaya dan inspiratif bagi masyarakat terima kasih juga atas kolaborasi positif bersama perumda air minum surya sembada
2,👏👏👏 selalu menjalin kerjasama yg baik dgn media. Keren 👍,selalu menjalin kerjasama yg baik dgn media keren
4,Selamatt dan semangattt indonesiaku😍🔥,selamatt dan semangattt indonesiaku
6,DM KU GA DIBALES KAK,dm ku ga dibales kak
9,salam buat tim spi min,salam buat tim spi min
10,Ini kenapa lagi surabaya barat daerah ptc airnya tiba2 mati parah banget udah kecil mati pula,ini kenapa lagi surabaya barat daerah ptc airnya tiba2 mati parah banget udah kecil mati pula
11,@baang_aziz 😂😂 waalaikumussalam bang aziz🔥❤️❤️,waalaikumussalam bang aziz
15,@pdamsuryasembada udah nyala kemarin jam 9 mlm an 😑😑😑,udah nyala kemarin jam 9 mlm an
17,@pdamsuryasembada 😂😂😂😂 sehat selalu min...👏🔥🔥🔥,sehat selalu min
19,Dm ku direspon dong kak,dm ku direspon dong kak


tokem

In [10]:
data['text'] = data['text'].astype(str).apply(word_tokenize)

In [11]:
def is_all_numbers(tokens):
    return all(re.fullmatch(r'\d+', tok) for tok in tokens)

data = data[~data['text'].apply(is_all_numbers)]


def normalize_mixed_number_token(token):
    if re.fullmatch(r'\d+', token):
        return "<num>"
    
    parts = re.findall(r'\d+|[a-zA-Z]+', token)

    if len(parts) == 1:
        return token

    parts = ["<num>" if p.isdigit() else p for p in parts]

    return " ".join(parts)


def normalize_tokens(token_list):
    normalized = []
    for tok in token_list:
        if tok.isdigit():
            normalized.append("<num>")
        elif re.search(r'\d', tok):
            normalized.append(normalize_mixed_number_token(tok))
        else:
            normalized.append(tok)
    return normalized

data["text"] = data["text"].apply(normalize_tokens)

In [12]:
data_partial = data.copy()
data_partial.to_csv('oct25_all_tokenized.csv', index=False)
data_partial


,post_id,comment_id,created_at,username,text,parent_comment_id,comments,timestamp
0,DQ51a5oEw2y,18075080645463218,2025-11-11 13:17:55 WIB,heframid,"[selamat, ulang, tahun, ke <num>, harian, sury...",NaN,Selamat ulang tahun ke-36 Harian Surya! 🎉 Semo...,1762841875
2,DQ51a5oEw2y,18542961421052542,2025-11-11 20:49:17 WIB,kharistiano28,"[selalu, menjalin, kerjasama, yg, baik, dgn, m...",NaN,👏👏👏 selalu menjalin kerjasama yg baik dgn medi...,1762868957
4,DQ5v5UsE4As,18005382170826298,2025-11-11 11:13:37 WIB,mhmdrizkyy12._,"[selamatt, dan, semangattt, indonesiaku]",NaN,Selamatt dan semangattt indonesiaku😍🔥,1762834417
6,DQ5ldh8E_V7,18078071812910678,2025-11-11 09:48:57 WIB,panggilnaw,"[dm, ku, ga, dibales, kak]",NaN,DM KU GA DIBALES KAK,1762829337
9,DQ6J9gWE8rk,17971535618800655,2025-11-11 16:04:18 WIB,baang_aziz,"[salam, buat, tim, spi, min]",NaN,salam buat tim spi min,1762851858
...,...,...,...,...,...,...,...,...
27646,Czvi08FPNgg,18259046566206582,2023-11-17 17:26:00 WIB,pratyaharasamadhi,"[ancen, tumanbanyu, mati, ra, kondo <num>, dis...",NaN,Ancen tuman...banyu mati ra kondo2 disik..usum...,1700216760
27647,Czvi08FPNgg,17980593236607765,2023-11-17 17:42:18 WIB,bapakduaanak,"[sak, jane, laporane, wong <num>, iki, bener <...",NaN,sak jane laporane wong² iki bener² di respon t...,1700217738
27648,Czvi08FPNgg,18027419773685068,2023-11-17 17:43:46 WIB,qenndra,"[kebiasaan, pdam, klo, mati, gk, woro <num>, d...",NaN,"Kebiasaan PDAM, klo mati gk woro2 dulu... Ada ...",1700217826
27649,Czvi08FPNgg,18059441194481795,2023-11-17 17:46:33 WIB,insomnila,"[sudah, lapor, tapi, belum, nyala, seharian]",NaN,Sudah lapor tapi belum nyala seharian,1700217993


formalize

In [13]:
with open("dictionary/dict_template3_doneig.json", "r", encoding="utf-8") as f:
    formal_dict3 = json.load(f)

with open("dictionary/dict4.json", "r", encoding="utf-8") as f:
    formal_dict4 = json.load(f)

formal_dict = {**formal_dict3, **formal_dict4}

def apply_formalization(tokens, formal_dict):
    
    new_tokens = []
    for tok in tokens:
        if tok in formal_dict:
            replacement = formal_dict[tok]
            new_tokens.extend(replacement.split())
            # new_tokens.append(formal_dict[tok]) 
        else:
            new_tokens.append(tok)
    return new_tokens

data_partial["text"] = data_partial["text"].apply(lambda tokens: apply_formalization(tokens, formal_dict))

delete stopwor

In [14]:
with open('stopwords_v4.txt', 'r', encoding='utf-8') as f:
    custom_stopwords = set([line.strip() for line in f if line.strip()])

def remove_stopwords(tokens, stopword_set):
    return [tok for tok in tokens if tok not in stopword_set and tok.strip() != ""]

data_partial["text"] = data_partial["text"].apply(lambda tokens: remove_stopwords(tokens, custom_stopwords))

stemmming

In [15]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.Stemmer.Stemmer import Stemmer
from Sastrawi.Stemmer.CachedStemmer import CachedStemmer
from Sastrawi.Stemmer.Cache.ArrayCache import ArrayCache
from Sastrawi.Dictionary.ArrayDictionary import ArrayDictionary

PROTECTED_WORDS = [
    'sememi', 'buran', 'kontrakan', 'medokan', 'kepatihan', 'keputih', 'perbaiki', 'perbaikan', 'menanggal', 'manukan', 'menganti',
]
#perbaikan,perbaiki ≠ baik ?
factory = StemmerFactory()
base_words = factory.get_words_from_file()
combined_words = list(set(base_words + PROTECTED_WORDS))
custom_dict = ArrayDictionary(combined_words)
stemmer = CachedStemmer(ArrayCache(), Stemmer(custom_dict))

def stem_tokens(tokens):
    if not isinstance(tokens, list):
        return tokens
    return [stemmer.stem(tok) for tok in tokens]

data_partial["text"] = data_partial["text"].apply(stem_tokens)

In [16]:
data_partial[['comments', 'text']].rename(columns={'comments': 'Sebelum', 'text': 'Sesudah'})

,Sebelum,Sesudah
0,Selamat ulang tahun ke-36 Harian Surya! 🎉 Semo...,"[ulang, hari, surya, moga, sumber, informasi, ..."
2,👏👏👏 selalu menjalin kerjasama yg baik dgn medi...,"[jalin, kerjasama, media, keren]"
4,Selamatt dan semangattt indonesiaku😍🔥,"[semangat, indonesia]"
6,DM KU GA DIBALES KAK,"[dm, balas]"
9,salam buat tim spi min,"[salam, tim, spi]"
...,...,...
27646,Ancen tuman...banyu mati ra kondo2 disik..usum...,"[tuman, mati, tidak, bilang, panas, kasihan, a..."
27647,sak jane laporane wong² iki bener² di respon t...,"[lapor, benar, respons, tidak, omong, mek, tin..."
27648,"Kebiasaan PDAM, klo mati gk woro2 dulu... Ada ...","[biasa, pdam, mati, tidak, umum, ada, anak, ba..."
27649,Sudah lapor tapi belum nyala seharian,"[lapor, nyala, hari]"


In [17]:
with open("dictionary/syn-dict.json", "r", encoding="utf-8") as f:
    synonym_dict = json.load(f)

antonym_df = pd.read_csv("dictionary/antonym-dict.csv")
antonym_dict = (
    antonym_df.dropna(subset=['antonym'])
              .set_index('token')['antonym']
              .to_dict()
)

negation_tokens = {'tidak', 'belum'}
def apply_negation_and_synonyms(token_list, negation_tokens, antonym_dict, synonym_dict):
    result = []
    skip_next = False
    for i, token in enumerate(token_list):
        if skip_next:
            skip_next = False 
            continue
        if token in negation_tokens and i + 1 < len(token_list):
            next_token = token_list[i + 1]
            if next_token in antonym_dict:
                result.append(antonym_dict[next_token])
            else:
                result.append(f'tidak_{next_token}')
            skip_next = True
        elif token in synonym_dict:
            result.append(synonym_dict[token])
        else:
            result.append(token)
    return result

data_partial["text"] = data_partial["text"].apply(
    lambda tokens: apply_negation_and_synonyms(tokens, negation_tokens, antonym_dict, synonym_dict)
)


In [20]:
with pd.option_context('display.max_colwidth', None):
    display(data_partial[['comments', 'text']].rename(columns={'comments': 'Sebelum', 'text': 'Sesudah'}).head(10))

,Sebelum,Sesudah
0,Selamat ulang tahun ke-36 Harian Surya! 🎉 Semoga terus menjadi sumber informasi terpercaya dan inspiratif bagi masyarakat. Terima kasih juga atas kolaborasi positif bersama Perumda Air Minum Surya Sembada!,"[ulang, hari, surya, moga, sumber, informasi, percaya, inspiratif, masyarakat, kolaborasi, positif, pdam, minum, surya, sembada]"
2,👏👏👏 selalu menjalin kerjasama yg baik dgn media. Keren 👍,"[jalin, kerjasama, media, keren]"
4,Selamatt dan semangattt indonesiaku😍🔥,"[semangat, indonesia]"
6,DM KU GA DIBALES KAK,"[dm, tanggap]"
9,salam buat tim spi min,"[salam, tim, spi]"
10,Ini kenapa lagi surabaya barat daerah ptc airnya tiba2 mati parah banget udah kecil mati pula,"[surabaya, barat, ptc, mati, parah, kecil, mati]"
11,@baang_aziz 😂😂 waalaikumussalam bang aziz🔥❤️❤️,[aziz]
15,@pdamsuryasembada udah nyala kemarin jam 9 mlm an 😑😑😑,[nyala]
17,@pdamsuryasembada 😂😂😂😂 sehat selalu min...👏🔥🔥🔥,[sehat]
19,Dm ku direspon dong kak,"[dm, tanggap]"


In [ ]:
test_data = data_partial.copy()
test_data = test_data.dropna(subset=["text"])
test_data["text"] = test_data["text"].astype(str).str.strip()
test_data = test_data[~test_data["text"].isin(["", "[]"])]

data_partial.to_csv("oct25_stemmed_all.csv", index=False)
data_partial

In [ ]:
test_data[['comments', 'text']].rename(columns={'comments': 'Sebelum', 'text': 'Sesudah'})


In [2]:
comments = pd.read_csv('oct25_stemmed_all.csv')

data_partial = comments.loc[:,['comment_id','created_at','username','text', 'comments']]
data_partial = data_partial[(data_partial['created_at'] > '2025-10-01') & (data_partial['created_at'] < '2025-11-01')]
import ast

data_partial['text'] = data_partial['text'].apply(ast.literal_eval)
data_partial

,comment_id,created_at,username,text,comments
10,17969564636959383,2025-10-29 23:44:51 WIB,n_aiiniiii,"[tidak_paham, harga, pasang, pdam]","Maaf nanya agak gak paham min , ini harga untu..."
11,17861299593523751,2025-10-30 05:38:39 WIB,raymundus_aryo,[ramai],Maw rame nih kdm min
12,18099012427673731,2025-10-31 04:47:10 WIB,opera8176,"[pdam, jalan, gresik, mati]","Min, PDAM Jalan Gresik mati. Gimana ini?"
13,18079780751119659,2025-10-31 10:05:24 WIB,matokebenny,"[tugas, daerah, ploso, timur, kali, cek, prema...",petugas wilayah ploso timur 3b itu tolong di S...
14,18105870148620168,2025-10-31 10:06:07 WIB,matokebenny,"[kali, kirim, komplain, langsung, pecat, tugas]",udah beberapa kali ngirim komplain langsung ha...
...,...,...,...,...,...
9760,18114324598556738,2025-10-16 01:37:13 WIB,mamaerisma30,"[pdam, bikin, nyala, susah, masak, begadang, l...",PDAM..PDAM...tolong dong jgn bikin tambah hidu...
9761,18080716520013483,2025-10-23 06:20:37 WIB,babyjotuhumena,"[pdam, hebat, ada, lambat, bayar, cepat, putus...","PDAM memang hebat, kl ada keterlambatan membay..."
9762,18410867584140778,2025-10-25 14:55:57 WIB,melongrus,"[daerah, lendang, nyala, mati]",Yaa ini diwilayah kami dilendang Beso dari kem...
9763,17897830962180200,2025-10-30 14:25:29 WIB,moh.asyari3112,"[daerah, lampung, bisa, adu]",Wilayah Lampung bisa pengaduan disini nga ya


tf(no IDF ig)

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
data_partial['processed_text'] = data_partial['text'].apply(
    lambda tokens: " ".join(tokens)
)
data_partial = data_partial[data_partial['processed_text'].str.strip() != ""]
vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    ngram_range=(1,1),
    min_df=0.0,
    max_df=0.8,
    use_idf=False,
    norm=None
)

vectors = vectorizer.fit_transform(data_partial['processed_text'])
feature_names = vectorizer.get_feature_names_out()
vectors_dense = vectors.toarray()

d:\Kuliah\PA\pdam-scraper\scrape_instagram\venv\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [4]:
print(f"Vector shape: {vectors.shape}")
print(f"nonzero: {vectors.nnz}")
row = vectors[0]
indices = row.indices
values = row.data

for i, v in zip(indices, values):
    print(feature_names[i], v) 

Vector shape: (1129, 1052)
nonzero: 5782
tidak_paham 1.0
harga 1.0
pasang 1.0
pdam 1.0


### Scenarios
There will be three different scenarios of vector dataset that will have its clustering runs:
1. Direct TF output
2. Threshold
3. Binary turned values
4. Threshold + Binary

#### Scenario 1

In [5]:
vectors = vectorizer.fit_transform(data_partial['processed_text'])
feature_names = vectorizer.get_feature_names_out()
vectors_dense = vectors.toarray()

vectors_dense_old = vectors_dense.copy()
df_old = pd.DataFrame(vectors_dense_old, columns=feature_names)
df_old.insert(0, 'comment_id', data_partial['comment_id'].values)
df_old.insert(1, 'comment', data_partial['processed_text'].values)
df_old.to_csv("oct25_tf_before_normalization_2.csv", index=False)
print(f"Old vector shape: {vectors_dense_old.shape}")

Old vector shape: (1129, 1052)


#### Scenario 2
*Threshold*
1. find the greatest valued feature pof each comment
2. half the value and use it as treshold for that comment
3. for any feature with a value in that comment, if it is less than half of it, then set it as zero

In [6]:
vectors_scene_2 = vectors_dense.copy()

for i in range(vectors_scene_2.shape[0]):
    row = vectors_scene_2[i]
    max_val = row.max()

    if max_val == 0:
        continue

    threshold = max_val / 2
    vectors_scene_2[i] = np.where(row >= threshold, row, 0.0)

non_zero_cols = np.any(vectors_scene_2 != 0, axis=0)
vectors_scene_2 = vectors_scene_2[:, non_zero_cols]
feature_names_scene_2 = feature_names[non_zero_cols]

print(f"New vector shape: {vectors_scene_2.shape}")
print(f"Features removed: {vectors_dense.shape[1] - vectors_scene_2.shape[1]}")

df_new2 = pd.DataFrame(vectors_scene_2, columns=feature_names_scene_2)
df_new2.insert(0, 'comment_id', data_partial['comment_id'].values)
df_new2.insert(1, 'comment', data_partial['processed_text'].values)
df_new2.to_csv("oct25_tf_after_threshold_2.csv", index=False)

New vector shape: (1129, 986)
Features removed: 66


In [7]:
# Mencari kata yang ada di feature_names tapi tidak ada di feature_names_scene_2
removed_features = set(feature_names) - set(feature_names_scene_2)
removed_list = list(removed_features)

print(f"Total fitur terhapus: {len(removed_list)}")
print(f"Contoh fitur terhapus: {removed_list[:66]}")

Total fitur terhapus: 66
Contoh fitur terhapus: ['simpan', 'tag', 'tidak_data', 'sesuai', 'kejar', 'tidak_laksana', 'cegat', 'super', 'jaring', 'wadah', 'sebentar', 'sedot', 'elus', 'tanda', 'istirahat', 'petang', 'kanan', 'sambut', 'tidak_anggur', 'utama', 'sengaja', 'klakah', 'kolektif', 'merek', 'fungsi', 'pusat', 'tidak_bocor', 'kendara', 'judi', 'tidak_liter', 'putar', 'agustus', 'alangkah', 'tidak_butek', 'beliau', 'steak', 'messenger', 'tidak_senang', 'malas', 'rincian', 'tidak_klarifikasi', 'bukti', 'tidak_bagi', 'istri', 'contoh', 'tidak_isi', 'murah', 'tidak_padan', 'foto', 'dinding', 'klarifikasi', 'tidak_rumah', 'pokok', 'main', 'sia', 'kamis', 'liat', 'tuntut', 'resto', 'tekan', 'kiri', 'tidak_bagus', 'hitam', 'depan', 'produksi', 'slogan']


#### Scenario 3
*Binary turned values*
1. for all feature with a non-zero value, ground them all to one
2. do this for all comments

In [8]:

vectors_scene_3 = (vectors_dense > 0).astype(int)
feature_names_scene_3 = feature_names
df_new3 = pd.DataFrame(vectors_scene_3, columns=feature_names_scene_3)
df_new3.insert(0, 'comment_id', data_partial['comment_id'].values)
df_new3.insert(1, 'comment', data_partial['processed_text'].values)
df_new3.to_csv("oct25_tf_after_binary_2.csv", index=False)
print(f"New vector shape: {vectors_scene_3.shape}")
print(f"Features removed: {vectors_dense.shape[1] - vectors_scene_3.shape[1]}")

New vector shape: (1129, 1052)
Features removed: 0


#### Scenario 4
*Threshold + Binary*

In [9]:
vectors_scene_4 = vectors_dense.copy()

for i in range(vectors_scene_4.shape[0]):
    row = vectors_scene_4[i]
    max_val = row.max()
    if max_val == 0:
        continue
    threshold = max_val / 2
    vectors_scene_4[i] = np.where(row >= threshold, row, 0.0)

non_zero_cols = np.any(vectors_scene_4 != 0, axis=0)
vectors_scene_4 = vectors_scene_4[:, non_zero_cols]
feature_names_scene_4 = feature_names[non_zero_cols]

vectors_scene_4 = (vectors_scene_4 > 0).astype(int)

print(f"New vector shape: {vectors_scene_4.shape}")
print(f"Features removed: {vectors_dense.shape[1] - vectors_scene_4.shape[1]}")

df_new4 = pd.DataFrame(vectors_scene_4, columns=feature_names_scene_4)
df_new4.insert(0, 'comment_id', data_partial['comment_id'].values)
df_new4.insert(1, 'comment', data_partial['processed_text'].values)
df_new4.to_csv("oct25_tf_after_threshold_binary_2.csv", index=False)

New vector shape: (1129, 986)
Features removed: 66


Attempt at valley tracing bruv

In [10]:
labeled_file_path = "dictionary/october_comments.xlsx"
comment_ids_ordered = data_partial['comment_id'].values

df_labels = pd.read_excel(labeled_file_path, sheet_name=1, dtype={"comment_id": str})
label_lookup = df_labels.set_index("comment_id")["Topik"]
gt_labels_text = np.array([label_lookup.get(str(cid), None) for cid in comment_ids_ordered])
labeled_mask = np.array([l is not None for l in gt_labels_text])
gt_labels_filtered = gt_labels_text[labeled_mask]
print(gt_labels_filtered[:10])
print(len(np.unique(gt_labels_filtered)))

['Pemasangan' 'Lainnya' 'Air Mati/Gangguan' 'Petugas' 'Petugas'
 'Debit Air' 'Air Mati/Gangguan' 'Lainnya' 'Pemasangan'
 'Air Mati/Gangguan']
11


d:\Kuliah\PA\pdam-scraper\scrape_instagram\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [11]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

VERSIONS = {
    "Scenario 1 (Direct TF Results)" : vectors_dense_old,
    "Scenario 2 (Threshold)":   vectors_scene_2,
    "Scenario 3 (Binary)":    vectors_scene_3,
    "Scenario 4 (Threshold + Binary)" : vectors_scene_4,
}

LINKAGE_METHODS = ['centroid', 'single', 'complete', 'average']

v_results = {}

for method in LINKAGE_METHODS:
    for ver_name, vec in VERSIONS.items():
        run_key = f"[{method.upper()}] {ver_name}"

        print(f"\n{'='*60}")
        print(f"Linkage: {method}  |  Scenario: {ver_name}")
        print(f"Shape: {vec.shape}")
        print('-'*60)

        N = vec.shape[0] - 1
        Z = linkage(vec, method=method, metric='euclidean')
        k_list = list(range(N, 1, -1))
        grand_mean = vec.mean(axis=0)

        sil         = []
        vw_list     = []
        vb_list     = []
        v_ratio     = []
        valley_k    = []
        valley_diff = []

        for k in k_list:
            labels          = fcluster(Z, t=k, criterion='maxclust')
            unique_clusters = np.unique(labels)

            delta_within  = 0.0
            delta_between = 0.0

            for cid in unique_clusters:
                pts = vec[labels == cid]
                ni  = pts.shape[0]
                if ni <= 1:
                    continue
                centroid  = pts.mean(axis=0)
                delta_sq  = np.sum(np.linalg.norm(pts - centroid, axis=1) ** 2) / (ni - 1)
                delta_within  += (ni - 1) * delta_sq
                delta_between += ni * np.linalg.norm(centroid - grand_mean) ** 2

            vw2 = delta_within  / (N - k) if (N - k) > 0 else np.nan
            vb2 = delta_between / (k - 1) if (k - 1) > 0 else np.nan
            vw_list.append(vw2)
            vb_list.append(vb2)
            v_ratio.append((vw2 / vb2) * 100 if (vb2 and vb2 > 0) else np.nan)

            sc = (silhouette_score(vec, labels, metric='euclidean')
                  if len(unique_clusters) >= 2 else np.nan)
            sil.append(sc)

        vw_list = np.array(vw_list)
        vb_list = np.array(vb_list)
        v_ratio = np.array(v_ratio)
        sil     = np.array(sil)

        # V-Ratio
        first    = np.diff(v_ratio)
        second   = np.diff(first)
        idx_curv = np.nanargmax(np.abs(second))
        k_curv   = k_list[idx_curv + 2]

        # Sil
        valid_idx = np.where(~np.isnan(sil))[0]
        k_sil     = k_list[valid_idx[np.nanargmax(sil[valid_idx])]] if len(valid_idx) else None

        # Valley-tracing
        for i in range(1, len(v_ratio) - 1):
            prev = v_ratio[i - 1]
            curr = v_ratio[i]
            nxt  = v_ratio[i + 1]
            if np.isnan(prev) or np.isnan(curr) or np.isnan(nxt):
                continue
            if prev >= curr and nxt > curr:
                valley_k.append(k_list[i])
                partial_diff = (prev + nxt) - (2 * curr)
                valley_diff.append(partial_diff)

        print("Valley-tracing candidate k values:", valley_k)

        # Accu
        max_diff = None
        max_k    = None
        accuracy = None
        if valley_k:
            paired   = sorted(zip(valley_diff, valley_k), reverse=True)
            max_diff = paired[0][0]
            max_k    = paired[0][1]

            if len(paired) >= 2:
                second_diff = paired[1][0]
                second_k    = paired[1][1]
                accuracy    = max_diff / second_diff if second_diff != 0 else float('inf')
                print(f"Best k by valley-tracing:  {max_k}")
                print(f"k Closest Value to max ∂:  {second_k}")
                print(f"Max ∂:                     {max_diff:.4f}")
                print(f"Second Max ∂:              {second_diff:.4f}")
                print(f"Acuuracy:              {accuracy:.4f}")
            else:
                print(f"Only one valley found — k: {max_k}, ∂: {max_diff:.4f}") #gk bisa uji akurasi

        if max_k is not None:
            best_cluster_labels = fcluster(Z, t=max_k, criterion='maxclust')
            cluster_labels_filtered = best_cluster_labels[labeled_mask]
            ari = adjusted_rand_score(gt_labels_filtered, cluster_labels_filtered)
        else:
            ari = np.nan
        print(f"ARI at best_k={max_k}: {ari:.4f}")
        
        v_results[run_key] = {
            'method'     : method,
            'scenario'   : ver_name,
            'Z'          : Z,
            'vec'        : vec,
            'k_list'     : k_list,
            'vw_list'    : vw_list,
            'vb_list'    : vb_list,
            'v_ratio'    : v_ratio,
            'sil'        : sil,
            'k_curv'     : k_curv,
            'k_sil'      : k_sil,
            'valley_k'   : valley_k,
            'valley_diff': max_diff,
            'best_k'     : max_k,
            'accuracy'   : accuracy,
            'ari'        : ari,
        }


Linkage: centroid  |  Scenario: Scenario 1 (Direct TF Results)
Shape: (1129, 1052)
------------------------------------------------------------
Valley-tracing candidate k values: [1003, 930, 901, 898, 894, 892, 890, 886, 880, 873, 869, 867, 863, 846, 841, 835, 833, 828, 825, 770, 728, 725, 719, 716, 714, 710, 687, 681, 679, 677, 664, 657, 650, 646, 643, 639, 595, 592, 590, 588, 586, 576, 568, 566, 563, 560, 551, 549, 545, 539, 537, 533, 528, 525, 521, 518, 512, 509, 490, 484, 480, 474, 471, 462, 452, 448, 444, 435, 433, 430, 426, 424, 408, 400, 395, 392, 362, 360, 347, 344, 329, 326, 318, 314, 305, 296, 283, 273, 267, 261, 233, 226, 217, 212, 198, 173, 120, 86, 78, 67, 18, 16, 12, 9, 7, 5]
Best k by valley-tracing:  18
k Closest Value to max ∂:  12
Max ∂:                     5042.8276
Second Max ∂:              653.1128
Acuuracy:              7.7212
ARI at best_k=18: -0.0037

Linkage: centroid  |  Scenario: Scenario 2 (Threshold)
Shape: (1129, 986)
------------------------------------

d:\Kuliah\PA\pdam-scraper\scrape_instagram\venv\Lib\site-packages\sklearn\metrics\cluster\_supervised.py:50: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(labels_pred)


Valley-tracing candidate k values: [1003, 929, 918, 913, 908, 902, 890, 862, 858, 854, 852, 780, 778, 771, 764, 746, 735, 732, 729, 727, 724, 720, 718, 713, 711, 705, 698, 696, 665, 662, 659, 656, 651, 644, 638, 636, 612, 605, 601, 579, 573, 569, 564, 553, 550, 545, 541, 506, 499, 494, 491, 489, 483, 478, 476, 472, 465, 459, 437, 430, 427, 424, 421, 413, 409, 404, 402, 398, 394, 390, 378, 370, 365, 360, 358, 356, 344, 340, 325, 323, 319, 315, 313, 310, 307, 302, 300, 298, 296, 293, 282, 279, 270, 267, 263, 258, 255, 246, 244, 241, 231, 229, 216, 213, 210, 208, 201, 199, 195, 190, 173, 171, 167, 152, 147, 137, 134, 132, 126, 117, 104, 100, 97, 94, 88, 78, 72, 62, 54, 48, 43, 24, 16, 10, 7, 5]
Best k by valley-tracing:  10
k Closest Value to max ∂:  16
Max ∂:                     6139.4908
Second Max ∂:              5042.7589
Acuuracy:              1.2175
ARI at best_k=10: -0.0036

Linkage: average  |  Scenario: Scenario 2 (Threshold)
Shape: (1129, 986)
-----------------------------------

In [13]:
feature_map = {
    "Scenario 1 (Direct TF Results)" : feature_names,
    "Scenario 2 (Threshold)":   feature_names_scene_2,
    "Scenario 3 (Binary)":    feature_names_scene_3,
    "Scenario 4 (Threshold + Binary)" : feature_names_scene_4,
}

def display_culster_details(tfidf_matrix, feature_names, labels, k, accuracy, ver_name="",):
    print(f"\n{'-'*30}")
    print(f"Cluster Details — {ver_name}  (k={k}) accuracy = {accuracy}")
    
    cluster_info = {}
    
    for cluster_id in range(1, k+1):
        indices = np.where(labels == cluster_id)[0]
        
        if len(indices) == 0:
            continue
        
        cluster_data = tfidf_matrix[indices]
        if hasattr(cluster_data, 'toarray'):
            cluster_data = cluster_data.toarray()
        cluster_mean = cluster_data.mean(axis=0)
        threshold = cluster_mean.max() / 2
        above_threshold = np.where(cluster_mean > threshold)[0]
        sorted_idx = above_threshold[np.argsort(-cluster_mean[above_threshold])]
        keywords = [(feature_names[i], cluster_mean[i]) for i in sorted_idx]
        
        cluster_info[cluster_id] = {
            'members': list(zip(
                data_partial['comment_id'].iloc[indices].tolist(),
                data_partial['processed_text'].iloc[indices].tolist()
            )),            'count': len(indices),
            'keywords': keywords
        }
    
    for cluster_id, info in cluster_info.items():
        print(f"\nCluster {cluster_id}  ({info['count']} members)")
        members_str = ", ".join([f"[{cid}] {txt}" for cid, txt in info['members']])
        print(f"  Members: {members_str}")  
        kw_str = ", ".join([f"{w}({s:.3f})" for w, s in info['keywords'][:10]])
        print(f"  Keywords: {kw_str if kw_str else '—'}")
    
    return cluster_info


for ver_name, res in v_results.items():
    k_used = res.get('best_k') or res['k_curv']
    k_used = int(k_used)
    labels_opt = fcluster(res['Z'], t=k_used, criterion='maxclust')
    scenario = res['scenario']
    features = feature_map.get(scenario, feature_names)
    accuracy = res.get('accuracy')

    display_culster_details(
        tfidf_matrix=res['vec'],
        feature_names=features,
        labels=labels_opt,
        k=k_used,
        accuracy=accuracy,
        ver_name=ver_name
    )


------------------------------
Cluster Details — [CENTROID] Scenario 1 (Direct TF Results)  (k=18) accuracy = 7.721220237694323

Cluster 1  (2 members)
  Members: [17909299632238845] pdam gayungsari barat daerah sekitar resto kampung steak mati nyala nyala mati nyala ada rumah mati tidak_bagi bingung tampung habis bantu pdam, [17899051386166643] pdam gayungsari barat daerah sekitar resto kampung steak ada bingung tampung rumah habis mati nyala nyala mati nyala ada rumah mati tidak_bagi bantu pdam
  Keywords: mati(3.000), nyala(3.000), pdam(2.000)

Cluster 2  (1111 members)
  Members: [17969564636959383] tidak_paham harga pasang pdam, [17861299593523751] ramai, [18099012427673731] pdam jalan gresik mati, [18079780751119659] tugas daerah ploso timur kali cek preman ketuk pagar ugal-ugalan kayu pentung preman pagar tetangga, [18105870148620168] kali kirim komplain langsung pecat tugas, [17891462820349844] pdam hati kecil nyala perbaik, [18081902578980188] kebraon susah aktivitas, [180618

In [ ]:
import re
import numpy as np

CURRENT_MONTH = "2023-12"
DOMINANT_THRESHOLD = 0.70

for ver_name, res in v_results.items():
    print(f"\n{'='*60}")
    
    linkage_method = ver_name.split(']')[0].strip('[').lower()
    scenario_str = res['scenario']
    scenario_match = re.search(r'Scenario (\d)', scenario_str)
    scenario_int = int(scenario_match.group(1)) if scenario_match else 1

    k_used = int(res.get('best_k') or res['k_curv'])
    labels_opt = fcluster(res['Z'], t=k_used, criterion='maxclust')
    features = feature_map.get(scenario_str, feature_names)
    accuracy = res['accuracy']

    unique_labels, counts = np.unique(labels_opt, return_counts=True)
    max_cluster_size = counts.max()
    max_cluster_pct = max_cluster_size / len(labels_opt)
    
    is_dominant_run = max_cluster_pct >= DOMINANT_THRESHOLD
    run_status = "dominant" if is_dominant_run else "done"
    print(f"Run: Scenario {scenario_int} | Linkage: {linkage_method} | k = {k_used}")
    print(f"Max cluster size: {max_cluster_pct:.1%} | Status: {run_status.upper()}")

    cluster_info = display_culster_details(res['vec'], features, labels_opt, k_used, ver_name)

    run_id = upsert_clustering_run(
        month=CURRENT_MONTH, 
        scenario=scenario_int, 
        linkage=linkage_method,
        optimal_k=k_used, 
        status=run_status, 
        dominant_threshold=float(max_cluster_pct),
        accuracy=accuracy,
        evaluation_score=res.get('ari', None)  # ← add this
    )
    clear_old_run_data(run_id)

    k_vals_clean = []
    v_ratio_clean = []
    
    for k, metric_val in zip(res['k_list'], res['v_ratio']):
        if not np.isnan(metric_val):
            k_vals_clean.append(int(k))
            v_ratio_clean.append(float(metric_val))
            
    if k_vals_clean:
        insert_valley_data(run_id, k_vals_clean, v_ratio_clean)
        print(f"Stored valley curve data for {len(k_vals_clean)} points.")
    else:
        print("Warning: No valid V-Ratio metrics to store for valley curve.")

    llm_results = {}
    if not is_dominant_run:
        print("\n--- Starting LLM Summarization (Groq Map-Reduce) ---")
        cluster_summaries = []
        
        for cluster_id, info in cluster_info.items():
            comments = [txt for cid, txt in info['members'] if pd.notna(txt) and str(txt).strip()]
            
            if len(comments) >= MIN_CLUSTER_SIZE:
                if len(comments) > MAX_COMMENTS_PER_CLUSTER:
                    comments = random.sample(comments, MAX_COMMENTS_PER_CLUSTER)
                
                keywords_str = ", ".join([w for w, score in info['keywords'][:10]])
                
                print(f"Summarizing & Labeling cluster {cluster_id} | Keywords: {keywords_str}")
                docs = [Document(page_content=c) for c in comments]
                
                try:
                    chain = load_summarize_chain(
                        llm,
                        chain_type="map_reduce",
                        map_prompt=map_prompt,
                        combine_prompt=combine_prompt,
                        verbose=False
                    )
                    summary = chain.invoke({
                        "input_documents": docs,
                        "keywords": keywords_str
                    })["output_text"].strip()
                    
                    joined_text = "\n".join(comments)
                    theme = llm.invoke(
                        theme_prompt.format(
                            responses=joined_text,
                            keywords=keywords_str,
                            domain_context=DOMAIN_CONTEXT
                        )
                    ).content.strip()

                    if len(theme.split()) > 5 or "\n" in theme or theme.lower().startswith("tidak"):
                        theme = "Misc / Minoritas"

                    cluster_summaries.append(summary)
                    llm_results[cluster_id] = {"tema": theme, "ringkasan": summary}
                    print(" ✓ Done")
                except Exception as e:
                    print(f" ✗ ERROR: {e}")
                    llm_results[cluster_id] = {"tema": "Error", "ringkasan": f"[ERROR: {e}]"}
                time.sleep(20)

        if cluster_summaries:
            print("\nGenerating Global Summary...")
            final_docs = [Document(page_content=s) for s in cluster_summaries]
            final_summary_chain = load_summarize_chain(llm, chain_type="stuff", prompt=reduce_prompt)
            final_summary = final_summary_chain.invoke({
                "input_documents": final_docs,
                "domain_context": DOMAIN_CONTEXT
            })["output_text"].strip()

            conn = get_connection()
            cur = conn.cursor()
            cur.execute("UPDATE clustering_runs SET notes = ? WHERE id = ?", (final_summary, run_id))
            conn.commit()
            conn.close()

    for cluster_id, info in cluster_info.items():
        cluster_size = info['count']
        is_this_cluster_dominant = (is_dominant_run and cluster_size == max_cluster_size)
        
        if is_dominant_run:
            display_name = f"Cluster {cluster_id}"
            summary_sentence = None
        elif cluster_size < MIN_CLUSTER_SIZE:
            display_name = "Misc / Minoritas"
            summary_sentence = "Komentar terlalu sedikit untuk dirangkum."
        else:
            llm_res = llm_results.get(cluster_id, {})
            display_name = llm_res.get("tema", f"Cluster {cluster_id}")
            summary_sentence = llm_res.get("ringkasan", None)

        top_keywords = [w for w, score in info['keywords'][:10]]

        db_cluster_id = insert_cluster(
            run_id=run_id, cluster_label=cluster_id, display_name=display_name,
            summary=summary_sentence, keywords=top_keywords,
            comment_count=cluster_size, is_dominant=is_this_cluster_dominant
        )

        instagram_ids = [str(cid) for cid, txt in info['members']]
        
        if instagram_ids:
            conn = get_connection()
            placeholders = ','.join(['?'] * len(instagram_ids))
            cur = conn.cursor()
            cur.execute(f"SELECT id FROM comments WHERE instagram_comment_id IN ({placeholders})", instagram_ids)
            sqlite_ids_to_map = [row[0] for row in cur.fetchall()]
            conn.close()
            
            map_comments_to_cluster(run_id, sqlite_ids_to_map, db_cluster_id)
        


Run: Scenario 1 | Linkage: centroid | k = 5
Max cluster size: 97.7% | Status: DOMINANT

------------------------------
Cluster Details —   (k=5) accuracy = [CENTROID] Scenario 1 (Direct TF Results)

Cluster 1  (2 members)
  Members: [18014108798016631] mohon mohon sememi sekitar mati dampak perbaik pipa bocor diameter kandang proses perbaik mohon doa perbaik distribusi langgan normal bisa mes tangki gratis call center tangki rumah mohon, [17998005560154593] bunga mohon ada perbaik pipa bocor diameter kandang proses perbaik mohon doa perbaik cepat distribusi langgan normal butuh desak bunga bisa mes tangki gratis call center tangki rumah mohon bunga
  Keywords: mohon, perbaik, tangki

Cluster 2  (217 members)
  Members: [17871099891033692] rumah mati, [17986142378563623] ada dng tapak siring payah, [17950236161724540] lebak rejo utara ada ganggu rumah sulit keluar, [17964103613559659] kira sampai karangpilang bukan libur laku monitoring debit mati, [18038810800623054] sememi mati ada, 

```
        # for cluster_id, info in cluster_info.items():
        #     comments = [txt for cid, txt in info['members'] if pd.notna(txt) and str(txt).strip()]
            
        #     if len(comments) >= MIN_CLUSTER_SIZE:
        #         if len(comments) > MAX_COMMENTS_PER_CLUSTER:
        #             comments = random.sample(comments, MAX_COMMENTS_PER_CLUSTER)
                
        #         print(f"Summarizing & Labeling cluster {cluster_id}...")
        #         docs = [Document(page_content=c) for c in comments]
                
        #         try:
        #             # Rangkuman (Map-Reduce)
        #             # chain = load_summarize_chain(llm, chain_type="map_reduce", map_prompt=map_prompt, combine_prompt=combine_prompt, verbose=False)
        #             # summary = chain.invoke(docs)["output_text"].strip()
        #             chain = load_summarize_chain(
        #                 llm,
        #                 chain_type="map_reduce",
        #                 map_prompt=map_prompt,
        #                 combine_prompt=combine_prompt,
        #                 verbose=False
        #             )
        #             # summary = chain.invoke({
        #             #     "input_documents": docs,
        #             #     "domain_context": DOMAIN_CONTEXT
        #             # })["output_text"].strip()
        #             summary = chain.invoke({
        #                 "input_documents": docs
        #             })["output_text"].strip()
                    
        #             # Tema
        #             joined_text = "\n".join(comments)
        #             # theme = llm.invoke(theme_prompt.format(responses=joined_text)).content.strip()
        #             # theme = llm.invoke(
        #             #     theme_prompt.format(responses=joined_text, domain_context=DOMAIN_CONTEXT)
        #             # ).content.strip()
        #             theme = llm.invoke(
        #                 theme_prompt.format(responses=joined_text, domain_context=DOMAIN_CONTEXT)
        #             ).content.strip()
                    
        #             if len(theme.split()) > 5 or "\n" in theme or theme.lower().startswith("tidak"):
        #                 theme = "Misc / Minoritas"

        #             cluster_summaries.append(summary)
        #             llm_results[cluster_id] = {"tema": theme, "ringkasan": summary}
        #             print(" ✓ Done")
        #         except Exception as e:
        #             print(f" ✗ ERROR: {e}")
        #             llm_results[cluster_id] = {"tema": "Error", "ringkasan": f"[ERROR: {e}]"}
        #         time.sleep(20)
```

In [ ]:
# def clear_non_october_ari():
#     conn = get_connection()
#     cur = conn.cursor()
    
#     # Set evaluation_score to NULL for all months except the one with ground truth
#     cur.execute("""
#         UPDATE clustering_runs 
#         SET evaluation_score = NULL 
#         WHERE month != '2025-10'
#     """)
    
#     affected = cur.rowcount
#     conn.commit()
#     conn.close()
#     print(f"Cleared evaluation_score for {affected} runs outside 2025-10.")

# clear_non_october_ari()

Cleared evaluation_score for 336 runs outside 2025-10.


In [ ]:
# import re
# import numpy as np

# CURRENT_MONTH = "2025-10"
# DOMINANT_THRESHOLD = 0.70

# for ver_name, res in v_results.items():
#     print(f"\n{'='*60}")
    
#     # 1. Extract Scenario & Linkage
#     linkage_method = ver_name.split(']')[0].strip('[').lower()
#     scenario_str = res['scenario']
#     scenario_match = re.search(r'Scenario (\d)', scenario_str)
#     scenario_int = int(scenario_match.group(1)) if scenario_match else 1

#     # 2. Clustering execution
#     k_used = int(res.get('best_k') or res['k_curv'])
#     labels_opt = fcluster(res['Z'], t=k_used, criterion='maxclust')
#     features = feature_map.get(scenario_str, feature_names)
#     accuracy = res.get('ari', None)  # ← now using ARI instead of valley accuracy

#     # 3. Polarity Check
#     unique_labels, counts = np.unique(labels_opt, return_counts=True)
#     max_cluster_size = counts.max()
#     max_cluster_pct = max_cluster_size / len(labels_opt)
    
#     is_dominant_run = max_cluster_pct >= DOMINANT_THRESHOLD
#     run_status = "dominant" if is_dominant_run else "done"
#     print(f"Run: Scenario {scenario_int} | Linkage: {linkage_method} | k = {k_used}")
#     print(f"Max cluster size: {max_cluster_pct:.1%} | Status: {run_status.upper()}")
#     print(f"ARI: {accuracy:.4f}" if accuracy is not None else "ARI: N/A")

#     # 4. Extract Keywords and Members
#     cluster_info = display_culster_details(res['vec'], features, labels_opt, k_used, ver_name)

#     # 5. DB Upsert - Run Metadata
#     run_id = upsert_clustering_run(
#         month=CURRENT_MONTH,
#         scenario=scenario_int,
#         linkage=linkage_method,
#         optimal_k=k_used,
#         status=run_status,
#         dominant_threshold=float(max_cluster_pct),
#         accuracy=res.get('accuracy', None),
#         evaluation_score=res.get('ari', None),
#     )
#     clear_old_run_data(run_id)

#     # 6. Insert Valley-Tracing Curve
#     k_vals_clean = []
#     v_ratio_clean = []
    
#     for k, metric_val in zip(res['k_list'], res['v_ratio']):
#         if not np.isnan(metric_val):
#             k_vals_clean.append(int(k))
#             v_ratio_clean.append(float(metric_val))
            
#     if k_vals_clean:
#         insert_valley_data(run_id, k_vals_clean, v_ratio_clean)
#         print(f"Stored valley curve data for {len(k_vals_clean)} points.")
#     else:
#         print("Warning: No valid V-Ratio metrics to store for valley curve.")

#     # 7. Save Clusters to DB (no LLM — placeholders only)
#     for cluster_id, info in cluster_info.items():
#         cluster_size = info['count']
#         is_this_cluster_dominant = (is_dominant_run and cluster_size == max_cluster_size)

#         if is_dominant_run:
#             display_name = f"Cluster {cluster_id}"
#             summary_sentence = None
#         elif cluster_size < MIN_CLUSTER_SIZE:
#             display_name = "Misc / Minoritas"
#             summary_sentence = "Komentar terlalu sedikit untuk dirangkum."
#         else:
#             display_name = f"Cluster {cluster_id}"  # placeholder, no LLM
#             summary_sentence = None                  # placeholder, no LLM

#         top_keywords = [w for w, score in info['keywords'][:10]]

#         db_cluster_id = insert_cluster(
#             run_id=run_id, cluster_label=cluster_id, display_name=display_name,
#             summary=summary_sentence, keywords=top_keywords,
#             comment_count=cluster_size, is_dominant=is_this_cluster_dominant
#         )

#         # Map comments to cluster in DB
#         instagram_ids = [str(cid) for cid, txt in info['members']]
        
#         if instagram_ids:
#             conn = get_connection()
#             placeholders = ','.join(['?'] * len(instagram_ids))
#             cur = conn.cursor()
#             cur.execute(f"SELECT id FROM comments WHERE instagram_comment_id IN ({placeholders})", instagram_ids)
#             sqlite_ids_to_map = [row[0] for row in cur.fetchall()]
#             conn.close()
            
#             map_comments_to_cluster(run_id, sqlite_ids_to_map, db_cluster_id)
        
#         print(f"Cluster {cluster_id} saved | size={cluster_size} | name='{display_name}'")

# print("\nAll runs pushed to DB successfully.")